<p align="center">
  <img src="archs/Self_Query.webp" alt="Self Query" width="1200">
</p>

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_cohere import CohereEmbeddings
from langchain_core.documents import Document
from langchain_classic.retrievers import SelfQueryRetriever
from langchain_classic.chains.query_constructor.base import AttributeInfo, StructuredQueryOutputParser, get_query_constructor_prompt
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

In [2]:
docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="A hacker discovers reality is a simulation and leads a rebellion against the machines controlling it.",
        metadata={"year": 1999, "director": "Lana Wachowski, Lilly Wachowski", "rating": 8.7, "genre": "science fiction"},
    ),
    Document(
        page_content="A young lion prince flees his kingdom only to learn the true meaning of responsibility and bravery.",
        metadata={"year": 1994, "rating": 8.5, "genre": "animated"},
    ),
    Document(
        page_content="Batman faces off against the Joker, a criminal mastermind who plunges Gotham into chaos.",
        metadata={"year": 2008, "director": "Christopher Nolan", "rating": 9.0, "genre": "action"},
    ),
    Document(
        page_content="A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.",
        metadata={"year": 2014, "director": "Christopher Nolan", "rating": 8.6, "genre": "science fiction"},
    ),
    Document(
        page_content="A hobbit embarks on an unexpected journey to reclaim a dwarf kingdom from a dragon.",
        metadata={"year": 2012, "director": "Peter Jackson", "rating": 7.8, "genre": "fantasy"},
    ),
    Document(
        page_content="A space cowboy and his misfit crew take on smuggling jobs while evading the law.",
        metadata={"year": 2005, "director": "Joss Whedon", "rating": 7.9, "genre": "science fiction"},
    ),
    Document(
        page_content="A masked vigilante fights crime in a dystopian future Britain ruled by a fascist government.",
        metadata={"year": 2005, "director": "James McTeigue", "rating": 8.2, "genre": "action"},
    ),
    Document(
        page_content="A chef rat secretly controls a young man to cook in a famous Paris restaurant.",
        metadata={"year": 2007, "director": "Brad Bird", "rating": 8.1, "genre": "animated"},
    ),
    Document(
        page_content="Two brothers use alchemy to search for the philosopher's stone to restore their bodies.",
        metadata={"year": 2003, "rating": 9.1, "genre": "animated"},
    ),
    Document(
        page_content="A soldier wakes up in another man's body and relives the last 8 minutes of a train explosion repeatedly.",
        metadata={"year": 2011, "director": "Duncan Jones", "rating": 7.5, "genre": "science fiction"},
    )
]

In [3]:
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=CohereEmbeddings(model="embed-v4.0"),
    collection_name="movie_collection",
    collection_metadata={"hnsw:space": "cosine"}
)

In [4]:
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated', 'fantasy']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="A 1-10 rating for the movie",
        type="float"
    ),
]

In [5]:
# This is not an user query but rather this tells the LLM what kinds of documents we have with us
document_content_description = "Brief summary of a movie"

prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
)

In [6]:
prompt

FewShotPromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, examples=[{'i': 1, 'data_source': '```json\n{{\n    "content": "Lyrics of a song",\n    "attributes": {{\n        "artist": {{\n            "type": "string",\n            "description": "Name of the song artist"\n        }},\n        "length": {{\n            "type": "integer",\n            "description": "Length of the song in seconds"\n        }},\n        "genre": {{\n            "type": "string",\n            "description": "The song genre, one of "pop", "rock" or "rap""\n        }}\n    }}\n}}\n```', 'user_query': 'What are songs by Taylor Swift or Katy Perry about teenage romance under 3 minutes long in the dance pop genre', 'structured_request': '```json\n{{\n    "query": "teenager love",\n    "filter": "and(or(eq(\\"artist\\", \\"Taylor Swift\\"), eq(\\"artist\\", \\"Katy Perry\\")), lt(\\"length\\", 180), eq(\\"genre\\", \\"pop\\"))"\n}}\n```'}, {'i': 2, 'data_source': '```json\n{{\n    "con

In [7]:
output_parser = StructuredQueryOutputParser.from_components()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.4,
    model_kwargs={
        "thinking_config": {"thinking_budget": 0}  # disable thinking
    }
)

In [8]:
output_parser = StructuredQueryOutputParser.from_components()

query_constructor = prompt | llm | output_parser

In [9]:
query_constructor.invoke("What are some sci-fi movies from the 90's directed by Luc Besson about taxi drivers")

StructuredQuery(query='taxi drivers', filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='genre', value='science fiction'), Comparison(comparator=<Comparator.GTE: 'gte'>, attribute='year', value=1990), Comparison(comparator=<Comparator.LTE: 'lte'>, attribute='year', value=1999), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='director', value='Luc Besson')]), limit=None)

In [10]:
retriever = SelfQueryRetriever(
    llm_chain=query_constructor,
    vectorstore=vector_store,
    structured_query_translator=ChromaTranslator(),
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5},
    use_original_query=False,       # If True, only metadata filters are used, semantic search is skipped. Useful when you want EXACT filter matches only
    verbose=True
)

In [11]:
retriever.invoke("What's a movie after 1990 but before 2005 that's all about toys, and preferably is animated")

[Document(metadata={'genre': 'animated', 'year': 1995}, page_content='Toys come alive and have a blast doing so'),
 Document(metadata={'genre': 'animated', 'rating': 8.5, 'year': 1994}, page_content='A young lion prince flees his kingdom only to learn the true meaning of responsibility and bravery.'),
 Document(metadata={'rating': 9.1, 'year': 2003, 'genre': 'animated'}, page_content="Two brothers use alchemy to search for the philosopher's stone to restore their bodies.")]

In [12]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [17]:
prompt = ChatPromptTemplate.from_template("""
You are a movie recommendation assistant.

Use ONLY the retrieved context.

Retrieved Context:
{context}

User Question:
{question}

Instructions:
1. Identify which retrieved movie best matches the request.
2. Explain why it matches.
3. Mention year, genre and rating if available.
4. If multiple movies match, rank them.
5. If none match, say that the context contains no matching movie.

Answer:
""")

In [18]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

main_chain = parallel_chain | prompt | llm | StrOutputParser()

In [19]:
print(main_chain.invoke("What's a movie after 1990 but before 2005 that's all about toys, and preferably is animated."))

Based on the retrieved context, here is the recommendation:

1. **Best Match:** The movie described as: *"Toys come alive and have a blast doing so."*
2. **Why it matches:** This description directly matches your request for a movie that is "all about toys," as it explicitly mentions toys coming alive. 
3. **Year, Genre, and Rating:** This information (year, genre, and rating) is not available in the provided context. 
4. **Ranking:** Only one movie in the context matches the description of being about toys, so no ranking is necessary.


In [20]:
print(main_chain.invoke("Who is Shinjan Saha?"))

Based on the retrieved context, there is no mention of "Shinjan Saha" or any movie description that relates to this query. Therefore, the context contains no matching movie.
